# Donut fine-tune — realistic InBody **270** v2 (issue #13) · Kaggle runner

Retrains Donut on the **new realistic 270** synthetic sheets to close the synthetic→real domain gap.
Fresh from `naver-clova-ix/donut-base`, **1 epoch**, **270-only** (570 deferred).

**Kaggle setup before running:**
1. *Settings → Accelerator* → **GPU T4 x2** (we pin to one GPU below — donut-base OOMs on GPU0 with both).
2. *Settings → Internet* → **On**.
3. *Add-ons → Secrets* → add **`GH_TOKEN`** (fine-grained, Contents: Read-only, scoped to QeekOw/InForm).

For an unattended run use **Save Version → Save & Run All (Commit)** — it runs headless (~90 min) and the checkpoint lands in the version output.
4. Upload the dataset to **Kaggle → Datasets → New Dataset**, then attach it here via **Add Input** (right panel). Cell 4 auto-detects it.


In [ ]:
# GPU + the two flags from prior runs: pin to one GPU (dual-T4 OOMs donut-base on
# GPU0) and enable expandable segments to avoid fragmentation OOMs.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Clone the issue-#13 branch (private repo -> GH_TOKEN secret) and install.
BRANCH = 'feat/realistic-synthetic-270'
from kaggle_secrets import UserSecretsClient
try:
    GH_TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
    REPO = f'https://{GH_TOKEN}@github.com/QeekOw/InForm.git'
except Exception:
    REPO = 'https://github.com/QeekOw/InForm.git'  # public fallback
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --branch $BRANCH --single-branch $REPO repo
%cd /kaggle/working/repo
!pip install -q -e '.[training]' gdown


In [ ]:
# Dataset attached as a Kaggle Dataset (Add Input, right panel) — no Drive/gdown.
# Find the sheets wherever Kaggle mounted them; if the input is still a .zip,
# extract it to /kaggle/tmp. Sets DATA_DIR to the folder holding the pngs.
import glob, os, shutil
pngs = glob.glob('/kaggle/input/**/*.png', recursive=True)
if not pngs:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No pngs or zip under /kaggle/input — click Add Input and attach your dataset.'
    shutil.unpack_archive(zips[0], '/kaggle/tmp')
    pngs = glob.glob('/kaggle/tmp/**/*.png', recursive=True)
assert pngs, 'No pngs found after extract — check the dataset contents.'
DATA_DIR = os.path.dirname(pngs[0])
print('DATA_DIR =', DATA_DIR, '| sheets:', len(pngs))


In [ ]:
# Fresh from donut-base, 1 epoch (memory: more epochs neither help nor hurt on
# this synthetic — 1 epoch for cost). Old checkpoint kept only as the 'before'
# baseline; we do NOT resume from it. batch 1 + grad-accum 4 fits the 2560x1920
# canvas on a T4; effective batch 4.
CHECKPOINT_DIR = '/kaggle/working/donut-270-v2'
!python -m inform.training.train \
  --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 1 --batch-size 1 --gradient-accumulation-steps 4 --learning-rate 3e-5


In [ ]:
# The checkpoint (model.safetensors + full processor) is under /kaggle/working,
# so it is included in the Save Version output — download it from the version's
# Output tab, then place it at D:/inform/checkpoints/donut-270-v2 for the local
# real-photo + held-out eval (inform.compare --skip-vlm).
!ls -la /kaggle/working/donut-270-v2


## After the run
- **Held-out synthetic:** `python -m inform.compare --data-dir <holdout_270_v2> --donut-checkpoint donut-270-v2 --skip-vlm`
- **Real-photo test:** same command against `D:/inform/data/real_holdout` — does it clear the old **0%**?
- Because we run exactly 1 epoch **to completion**, `train.py` saves the processor itself — no manual processor-attach needed (that gotcha was only for early-interrupt runs).
